# Lab 5: Genetic Algorithms

**Module:** Artificial Intelligence  
**Topic:** Evolutionary Computing

---

### Learning Objectives

By the end of this lab, you will be able to:

1. Explain how a Genetic Algorithm (GA) works
2. Implement **selection**, **crossover**, and **mutation** from scratch using NumPy
3. Apply a GA to find the maximum of a mathematical function
4. Track and visualise **fitness evolution** over generations
5. Tune the key GA parameters (population size, mutation rate, generations)

---

## Background

### What is a Genetic Algorithm?

A **Genetic Algorithm (GA)** is a search and optimisation method inspired by natural selection.  
It maintains a **population** of candidate solutions ("chromosomes") and improves them over **generations** using three operators:

| Operator | Biological analogy | What it does |
|---|---|---|
| **Selection** | Survival of the fittest | Prefer solutions with higher fitness for reproduction |
| **Crossover** | Reproduction / recombination | Combine parts of two parents to create offspring |
| **Mutation** | Random genetic mutation | Randomly alter genes to maintain diversity |

### The GA Loop

```
1. Initialise a random population
2. Evaluate fitness of each individual
3. Repeat until stopping condition:
   a. Select parents (e.g. tournament selection)
   b. Apply crossover to produce offspring
   c. Apply mutation to offspring
   d. Evaluate fitness of offspring
   e. Replace population with offspring
4. Return best solution found
```

### Chromosome Representation

In this lab each individual is a **real-valued vector** (array of floats representing variables to optimise).

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---

## Problem: Maximise a Function

We want to find the value of `x` in the range `[-5, 5]` that **maximises**:

$$f(x) = -x^2 + 4x + 4$$

The true maximum is at **x = 2**, giving f(2) = 8.  
We will let the GA discover this purely through evolution — no calculus required.

In [ ]:
def fitness(x):
    """
    Fitness function to maximise: f(x) = -x^2 + 4x + 4

    Parameters
    ----------
    x : float or array-like

    Returns
    -------
    float or array  Fitness value(s).
    """
    return -x**2 + 4*x + 4


# Visualise the function
x_plot = np.linspace(-5, 5, 300)
plt.figure(figsize=(8, 4))
plt.plot(x_plot, fitness(x_plot), color='royalblue', linewidth=2)
plt.axvline(x=2, color='red', linestyle='--', label='True maximum (x=2)')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Fitness function: f(x) = -x² + 4x + 4')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"True maximum: f(2) = {fitness(2):.4f}")

---

## Exercise 1: Initialise Population

Create a population of `n` random individuals, each a single float in `[x_min, x_max]`.

In [ ]:
def init_population(pop_size, x_min, x_max):
    """
    Initialise a random population.

    Parameters
    ----------
    pop_size : int    Number of individuals.
    x_min    : float  Lower bound of search space.
    x_max    : float  Upper bound of search space.

    Returns
    -------
    numpy array of shape (pop_size,)
    """
    return np.random.uniform(x_min, x_max, size=pop_size)


# Test
pop = init_population(10, -5, 5)
print("Initial population:", np.round(pop, 3))
print("Fitness values:    ", np.round(fitness(pop), 3))

---

## Exercise 2: Tournament Selection

Pick `k` individuals at random and return the one with the **highest fitness**.

In [ ]:
def tournament_select(population, fitness_fn, k=3):
    """
    Tournament selection: pick the best individual from k random candidates.

    Parameters
    ----------
    population : array   Current population.
    fitness_fn : callable  Fitness function.
    k          : int     Tournament size.

    Returns
    -------
    float  The selected individual.
    """
    indices = np.random.choice(len(population), size=k, replace=False)
    contestants = population[indices]
    best_idx = np.argmax(fitness_fn(contestants))
    return contestants[best_idx]


# Test
pop = init_population(20, -5, 5)
parent = tournament_select(pop, fitness)
print(f"Selected parent: x = {parent:.4f},  f(x) = {fitness(parent):.4f}")

---

## Exercise 3: Crossover

For real-valued chromosomes, **arithmetic crossover** blends two parents:

$$\text{child} = \alpha \cdot p_1 + (1 - \alpha) \cdot p_2, \quad \alpha \sim U(0, 1)$$

In [ ]:
def crossover(parent1, parent2):
    """
    Arithmetic crossover: blend two parents with a random weight.

    Parameters
    ----------
    parent1, parent2 : float  Parent individuals.

    Returns
    -------
    float  Offspring value.
    """
    alpha = np.random.random()
    return alpha * parent1 + (1 - alpha) * parent2


# Test
p1, p2 = 1.0, 3.0
child = crossover(p1, p2)
print(f"Parent 1: {p1},  Parent 2: {p2},  Child: {child:.4f}")

---

## Exercise 4: Mutation

With probability `mutation_rate`, add a small Gaussian noise to the individual and clip to bounds.

In [ ]:
def mutate(individual, mutation_rate, x_min, x_max, sigma=0.5):
    """
    Gaussian mutation: add random noise with probability mutation_rate.

    Parameters
    ----------
    individual    : float  The individual to mutate.
    mutation_rate : float  Probability of mutation (0 to 1).
    x_min, x_max  : float  Search space bounds (for clipping).
    sigma         : float  Standard deviation of Gaussian noise.

    Returns
    -------
    float  (possibly mutated) individual.
    """
    if np.random.random() < mutation_rate:
        individual = individual + np.random.normal(0, sigma)
        individual = np.clip(individual, x_min, x_max)
    return individual


# Test
val = 2.0
for _ in range(5):
    print(f"  mutate({val:.2f}) -> {mutate(val, mutation_rate=1.0, x_min=-5, x_max=5):.4f}")

---

## Exercise 5: The Full Genetic Algorithm

Assemble all the operators into the main GA loop.

In [ ]:
def genetic_algorithm(
    fitness_fn,
    pop_size=50,
    generations=100,
    x_min=-5.0,
    x_max=5.0,
    mutation_rate=0.1,
    tournament_k=3
):
    """
    Simple real-valued Genetic Algorithm.

    Parameters
    ----------
    fitness_fn    : callable  Function to maximise.
    pop_size      : int       Number of individuals per generation.
    generations   : int       Number of generations to run.
    x_min, x_max  : float     Search space bounds.
    mutation_rate : float     Probability of mutating each offspring.
    tournament_k  : int       Tournament size for selection.

    Returns
    -------
    best_x        : float   Best x found.
    best_fitness  : float   Fitness of best x.
    history       : list    Best fitness value at each generation.
    """
    # 1. Initialise
    population = init_population(pop_size, x_min, x_max)
    history = []

    for gen in range(generations):
        # 2. Evaluate
        fits = fitness_fn(population)
        best_idx = np.argmax(fits)
        history.append(fits[best_idx])

        # 3. Create next generation
        new_population = []
        for _ in range(pop_size):
            # Select two parents
            p1 = tournament_select(population, fitness_fn, k=tournament_k)
            p2 = tournament_select(population, fitness_fn, k=tournament_k)
            # Crossover
            child = crossover(p1, p2)
            # Mutation
            child = mutate(child, mutation_rate, x_min, x_max)
            new_population.append(child)

        population = np.array(new_population)

    # Final evaluation
    fits = fitness_fn(population)
    best_idx = np.argmax(fits)
    return population[best_idx], fits[best_idx], history


# Run the GA
best_x, best_f, history = genetic_algorithm(
    fitness_fn=fitness,
    pop_size=50,
    generations=100,
    x_min=-5.0,
    x_max=5.0,
    mutation_rate=0.1
)

print(f"GA result:    x = {best_x:.6f}")
print(f"Fitness:  f(x) = {best_f:.6f}")
print(f"True max:  f(2) = {fitness(2):.6f}")

---

## Exercise 6: Visualise Fitness Evolution

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history, color='darkorange', linewidth=2)
plt.axhline(y=fitness(2), color='red', linestyle='--', label=f'True maximum = {fitness(2)}')
plt.xlabel('Generation')
plt.ylabel('Best fitness')
plt.title('Genetic Algorithm — Fitness Evolution')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## Exercise 7: Population Diversity Over Time

Track how the population spreads (or converges) across generations.

In [ ]:
def genetic_algorithm_tracked(fitness_fn, pop_size=50, generations=100,
                               x_min=-5.0, x_max=5.0, mutation_rate=0.1, tournament_k=3):
    """Same GA but also records population snapshots at key generations."""
    population = init_population(pop_size, x_min, x_max)
    snapshots = {}   # generation -> population copy
    history = []

    for gen in range(generations):
        fits = fitness_fn(population)
        history.append(np.max(fits))

        if gen in (0, 9, 24, 49, 99):
            snapshots[gen] = population.copy()

        new_population = []
        for _ in range(pop_size):
            p1 = tournament_select(population, fitness_fn, k=tournament_k)
            p2 = tournament_select(population, fitness_fn, k=tournament_k)
            child = crossover(p1, p2)
            child = mutate(child, mutation_rate, x_min, x_max)
            new_population.append(child)
        population = np.array(new_population)

    fits = fitness_fn(population)
    return population[np.argmax(fits)], np.max(fits), history, snapshots


best_x, best_f, history, snapshots = genetic_algorithm_tracked(
    fitness, pop_size=50, generations=100, x_min=-5, x_max=5, mutation_rate=0.1
)

x_plot = np.linspace(-5, 5, 300)
fig, axes = plt.subplots(1, len(snapshots), figsize=(16, 4), sharey=True)

for ax, (gen, pop) in zip(axes, snapshots.items()):
    ax.plot(x_plot, fitness(x_plot), color='royalblue', alpha=0.4)
    ax.scatter(pop, fitness(pop), color='darkorange', s=20, zorder=5)
    ax.axvline(x=2, color='red', linestyle='--', alpha=0.6)
    ax.set_title(f'Gen {gen + 1}')
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('f(x)')
fig.suptitle('Population Distribution Across Generations', fontsize=13)
plt.tight_layout()
plt.show()

---

## Exercise 8: Effect of Mutation Rate

Compare how different mutation rates affect convergence speed and final fitness.

In [ ]:
mutation_rates = [0.01, 0.1, 0.3, 0.6]
colors = ['green', 'darkorange', 'blue', 'red']

plt.figure(figsize=(10, 5))
for mr, col in zip(mutation_rates, colors):
    np.random.seed(42)   # same start for fair comparison
    _, _, hist = genetic_algorithm(
        fitness, pop_size=50, generations=100,
        x_min=-5, x_max=5, mutation_rate=mr
    )
    plt.plot(hist, color=col, label=f'mutation_rate={mr}')

plt.axhline(y=fitness(2), color='black', linestyle='--', label=f'True max={fitness(2)}')
plt.xlabel('Generation')
plt.ylabel('Best fitness')
plt.title('Effect of Mutation Rate on Convergence')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## Summary

In this lab you built a complete **Genetic Algorithm from scratch** using only NumPy:

| Component | Implementation |
|---|---|
| **Population** | `init_population` — random floats in `[x_min, x_max]` |
| **Fitness** | `fitness(x) = -x² + 4x + 4` |
| **Selection** | `tournament_select` — best of k random candidates |
| **Crossover** | `crossover` — arithmetic blend of two parents |
| **Mutation** | `mutate` — Gaussian noise with probability `mutation_rate` |

### Key Observations

- The GA converges near the true maximum **x = 2, f(2) = 8** without any gradient information.
- **Low mutation rates** converge faster but may get stuck.
- **High mutation rates** keep diversity but slow down convergence.
- Population plots show individuals clustering around the optimum over time.